# Step 2A – Dynamic Data Generation (Statistical Simulation)
Mô phỏng **1000 đơn hàng (order)** của **350 khách hàng** mua sản phẩm chăm sóc cá nhân Pharmacity.

- Mỗi khách hàng có thông tin **cố định** (tên, tuổi, giới tính, thành phố) và có thể mua nhiều đơn.
- Mỗi order là 1 dòng; `ProductID` là khóa để merge với bảng sản phẩm ở Step 3.

In [1]:
import random
import numpy as np
import pandas as pd
from faker import Faker

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
fake = Faker("vi_VN")
Faker.seed(SEED)

N_ORDERS = 1000      # tổng số đơn hàng
N_CUSTOMERS = 350    # số khách hàng riêng biệt

## 1. Đọc bảng sản phẩm (chỉ cần ProductID và Price)

In [2]:
products_df = pd.read_csv("pharmacity_personal_care.csv")[["ProductID", "Price"]]
print(products_df.shape)
products_df.head()

(445, 2)


,ProductID,Price
0,P07609,49500
1,P00326,18000
2,P26024,68250
3,P03828,116000
4,P17958,270100


## 2. Tạo bảng 350 khách hàng
Mỗi khách hàng chỉ tạo **1 lần**, nên tên/tuổi/giới tính/thành phố không bị đổi giữa các đơn.

In [3]:
customer_ids = [f"C{i:04d}" for i in range(1, N_CUSTOMERS + 1)]

# Giới tính trước, rồi tạo tên theo giới tính cho hợp lý
genders = random.choices(["Nam", "Nữ"], weights=[0.45, 0.55], k=N_CUSTOMERS)

def make_name(gender):
    if gender == "Nam":
        return f"{fake.last_name()} {random.choice(['Văn', 'Hữu', 'Minh', 'Quốc'])} {fake.first_name_male()}"
    return f"{fake.last_name()} {random.choice(['Thị', 'Ngọc', 'Thu', 'Mỹ'])} {fake.first_name_female()}"

customer_names = [make_name(g) for g in genders]

# Tuổi theo phân phối chuẩn (trung bình 30, độ lệch chuẩn 10), giới hạn 18-70
ages = np.random.normal(loc=30, scale=10, size=N_CUSTOMERS).round().astype(int)
ages = np.clip(ages, 18, 70)

cities = random.choices(
    ["TP. Hồ Chí Minh", "Hà Nội", "Đà Nẵng", "Cần Thơ", "Hải Phòng"],
    weights=[0.40, 0.30, 0.10, 0.10, 0.10],
    k=N_CUSTOMERS,
)

customers_df = pd.DataFrame({
    "CustomerID": customer_ids,
    "CustomerName": customer_names,
    "Age": ages,
    "Gender": genders,
    "City": cities,
})
customers_df.head()

,CustomerID,CustomerName,Age,Gender,City
0,C0001,Trần Thu Ngọc,35,Nữ,TP. Hồ Chí Minh
1,C0002,Vũ Văn Quang,29,Nam,TP. Hồ Chí Minh
2,C0003,Phạm Văn Hoàng,36,Nam,TP. Hồ Chí Minh
3,C0004,Trần Minh Tùng,45,Nam,TP. Hồ Chí Minh
4,C0005,Trần Ngọc Duyên,28,Nữ,TP. Hồ Chí Minh


## 3. Gán 1000 order cho 350 khách hàng
- 350 order đầu: mỗi khách có **ít nhất 1 đơn** (đảm bảo đủ đúng 350 khách).
- 650 order còn lại: chọn khách ngẫu nhiên → một số khách sẽ mua nhiều lần.

In [4]:
order_customer_ids = customer_ids + random.choices(customer_ids, k=N_ORDERS - N_CUSTOMERS)
random.shuffle(order_customer_ids)
len(order_customer_ids)

1000

## 4. Tạo thông tin từng order (sản phẩm, số lượng, ngày mua, thanh toán)

In [5]:
# Mỗi order chọn ngẫu nhiên 1 sản phẩm (có thể trùng sản phẩm giữa các order)
chosen_products = products_df.sample(n=N_ORDERS, replace=True, random_state=SEED).reset_index(drop=True)

quantities = np.random.choice([1, 2, 3, 4, 5], size=N_ORDERS, p=[0.50, 0.25, 0.15, 0.07, 0.03])

payment_methods = random.choices(
    ["Tiền mặt", "Thẻ ngân hàng", "Ví điện tử", "COD"],
    weights=[0.20, 0.25, 0.25, 0.30],
    k=N_ORDERS,
)

purchase_dates = pd.to_datetime(
    np.random.choice(pd.date_range("2026-01-01", "2026-09-30"), size=N_ORDERS)
)

orders_df = pd.DataFrame({
    "CustomerID": order_customer_ids,
    "ProductID": chosen_products["ProductID"],
    "Quantity": quantities,
    "TotalAmount": chosen_products["Price"] * quantities,   # = Giá x Số lượng
    "PurchaseDate": purchase_dates,
    "PaymentMethod": payment_methods,
})
orders_df.head()

,CustomerID,ProductID,Quantity,TotalAmount,PurchaseDate,PaymentMethod
0,C0180,P18999,3,583200,2026-04-24,COD
1,C0124,P13420,2,122000,2026-05-25,COD
2,C0199,P12848,1,189660,2026-01-26,Thẻ ngân hàng
3,C0037,P30966,4,160000,2026-05-16,Ví điện tử
4,C0236,P21174,1,203000,2026-08-20,Tiền mặt


## 5. Ghép thông tin khách hàng vào order và sắp xếp theo ngày
Dùng `pd.merge(..., how="left")`: giữ nguyên 1000 order, mỗi order lấy thêm thông tin của khách tương ứng.

In [6]:
Order_df = orders_df.merge(customers_df, on="CustomerID", how="left")

# Sắp xếp theo ngày mua rồi mới đánh OrderID (đơn cũ có ID nhỏ hơn)
Order_df = Order_df.sort_values("PurchaseDate").reset_index(drop=True)
Order_df["OrderID"] = [f"ORD{i:05d}" for i in range(1, N_ORDERS + 1)]

# Sắp xếp lại thứ tự cột cho dễ đọc
Order_df = Order_df[[
    "OrderID", "CustomerID", "CustomerName", "Age", "Gender", "City",
    "ProductID", "Quantity", "TotalAmount", "PurchaseDate", "PaymentMethod",
]]
Order_df.head()

,OrderID,CustomerID,CustomerName,Age,Gender,City,ProductID,Quantity,TotalAmount,PurchaseDate,PaymentMethod
0,ORD00001,C0214,Nguyễn Quốc Khoa,37,Nam,Hải Phòng,P10412,3,57000,2026-01-01,Ví điện tử
1,ORD00002,C0020,Trần Hữu Thành,18,Nam,Đà Nẵng,P18082,3,123000,2026-01-01,Thẻ ngân hàng
2,ORD00003,C0144,Bùi Thị Xuân,32,Nữ,TP. Hồ Chí Minh,P29620,1,194250,2026-01-01,Tiền mặt
3,ORD00004,C0253,Nguyễn Minh Huy,51,Nam,Hà Nội,P28349,1,110000,2026-01-02,COD
4,ORD00005,C0138,Nguyễn Mỹ Dương,27,Nữ,Cần Thơ,P25083,3,37500,2026-01-02,Tiền mặt


## 6. Kiểm tra chất lượng trước khi sang Step 3

In [7]:
print("Kích thước:", Order_df.shape)
print("Số order:", Order_df["OrderID"].nunique())
print("Số khách hàng riêng biệt:", Order_df["CustomerID"].nunique())
print("Tổng giá trị thiếu (NaN):", Order_df.isna().sum().sum())

# Thông tin mỗi khách phải nhất quán (mỗi CustomerID chỉ có 1 tên, 1 tuổi, 1 giới tính, 1 thành phố)
check = Order_df.groupby("CustomerID")[["CustomerName", "Age", "Gender", "City"]].nunique()
print("Thông tin khách nhất quán:", (check == 1).all().all())

# Mọi ProductID phải có trong bảng sản phẩm (để merge ở Step 3 không bị mất dữ liệu)
print("ProductID đều có trong bảng sản phẩm:", Order_df["ProductID"].isin(products_df["ProductID"]).all())

# Tổng tiền đúng = giá x số lượng
price_check = Order_df.merge(products_df, on="ProductID")
print("TotalAmount đúng:", (price_check["TotalAmount"] == price_check["Price"] * price_check["Quantity"]).all())

print("\nSố order mỗi khách:")
print(Order_df["CustomerID"].value_counts().describe().round(2))
print()
Order_df.dtypes

Kích thước: (1000, 11)
Số order: 1000
Số khách hàng riêng biệt: 350
Tổng giá trị thiếu (NaN): 0
Thông tin khách nhất quán: True
ProductID đều có trong bảng sản phẩm: True
TotalAmount đúng: True

Số order mỗi khách:
count    350.00
mean       2.86
std        1.40
min        1.00
25%        2.00
50%        3.00
75%        4.00
max        7.00
Name: count, dtype: float64



OrderID                     str
CustomerID                  str
CustomerName                str
Age                       int64
Gender                      str
City                        str
ProductID                   str
Quantity                  int64
TotalAmount               int64
PurchaseDate     datetime64[us]
PaymentMethod               str
dtype: object

In [8]:
# Phân phối nhanh để chắc là hợp lý
print(Order_df["Gender"].value_counts(normalize=True).round(2))
print(Order_df["City"].value_counts(normalize=True).round(2))
print(Order_df["PaymentMethod"].value_counts(normalize=True).round(2))
print(Order_df["PurchaseDate"].min(), "->", Order_df["PurchaseDate"].max())

Gender
Nữ     0.55
Nam    0.45
Name: proportion, dtype: float64
City
TP. Hồ Chí Minh    0.41
Hà Nội             0.28
Hải Phòng          0.11
Cần Thơ            0.10
Đà Nẵng            0.09
Name: proportion, dtype: float64
PaymentMethod
COD              0.31
Thẻ ngân hàng    0.24
Ví điện tử       0.24
Tiền mặt         0.21
Name: proportion, dtype: float64
2026-01-01 00:00:00 -> 2026-09-30 00:00:00


## 7. Xuất file

In [9]:
# utf-8-sig để mở bằng Excel không bị lỗi font tiếng Việt
Order_df.to_csv("pharmacity_simulated_transactions.csv", index=False, encoding="utf-8-sig")
print("Đã lưu pharmacity_simulated_transactions.csv")

Đã lưu pharmacity_simulated_transactions.csv


## Ghi chú cho Step 3 (merge)
- Khóa merge: `ProductID`. Bảng order **không lặp lại** các cột tĩnh (ProductName, Category, Brand, Price…) → khi merge với Unified Static Master Table sẽ không bị trùng cột (`_x`, `_y`).
- Nên dùng **Left Join** với bảng order ở bên trái: giữ đủ 1000 order, mỗi order lấy thêm thông tin sản phẩm. (Sản phẩm Sephora trong master table không có order nào, nên Inner/Right/Outer sẽ cho kết quả khác – cần giải thích trong báo cáo.)